In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot  as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

In [2]:
def pre_process(filename):
    # Load data
    df = pd.read_csv(filename, index_col='id')
    # Drop irrelevant columns
    df.drop(['suburb_lat','date_sold','ethnic_breakdown','suburb_elevation','suburb','region','avg_years_held',
        'cash_rate','suburb_lng','suburb_sqkm','suburb_population','suburbpopulation','public_housing_pct','postcode',
        'nearest_train_station','highlights_attractions', 'ideal_for', 'traffic', 'public_transport', 'affordability_rental',
        # 'time_to_cbd_public_transport_town_hall_st','time_to_cbd_driving_town_hall_st',
        'suburb_median_income', 'property_inflation_index',
        'affordability_buying', 'nature', 'noise', 'things_to_see_do', 'family_friendliness', 'pet_friendliness', 'safety', 'overall_rating'
    ], axis=1, inplace=True)

    df['commute_time'] = (df['time_to_cbd_public_transport_town_hall_st']+df['time_to_cbd_driving_town_hall_st'])/2
    df['commute_time'] = df['commute_time'].fillna(df['commute_time'].mean())
    df=df.drop(['time_to_cbd_public_transport_town_hall_st','time_to_cbd_driving_town_hall_st'],axis=1)

    # Add new features
    df['num_of_rooms'] = df['num_bed'] + df['num_bath']
    df['inverse_cbd_distance'] = 1 / (df['km_from_cbd'] + 1)
    # df['avg_room_size'] = (df['property_size'] / 1+df['num_of_rooms']).round()
    
    # Drop redundant
    df.drop(['km_from_cbd'], axis=1, inplace=True)

    return df

In [3]:
train_df = pre_process("train.csv")
test_df = pre_process("test.csv")
df = pre_process("train.csv")

In [4]:
def feature_engineering(df):
    #Split the Features
    discrete_threshold = 25
    numerical_discrete = []
    numerical_continuous = []
    categorical_features = []
    exclude_cols = ['id','type']
    for col in df.columns:
        if col not in exclude_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                unique_vals = df[col].nunique()
                if unique_vals <= discrete_threshold :
                    numerical_discrete.append(col)
                else:
                    numerical_continuous.append(col)
            else:
                categorical_features.append(col)
        
    print("Discrete numerical features:")
    print(numerical_discrete)
    print("\nContinuous numerical features:")
    print(numerical_continuous)
    print("\nCategorical features:")
    print(categorical_features )
    
    return numerical_discrete,numerical_continuous,categorical_features

In [6]:
discrete_features,continuous_features,categorical_features=feature_engineering(train_df)
features = train_df.drop('type' ,axis=1).columns
target = 'type'
print(features)

Discrete numerical features:
['num_bath', 'num_bed', 'num_parking']

Continuous numerical features:
['price', 'property_size', 'suburb_median_house_price', 'median_house_rent_per_week', 'suburb_median_apartment_price', 'median_apartment_rent_per_week', 'commute_time', 'num_of_rooms', 'inverse_cbd_distance']

Categorical features:
[]
Index(['price', 'num_bath', 'num_bed', 'num_parking', 'property_size',
       'suburb_median_house_price', 'median_house_rent_per_week',
       'suburb_median_apartment_price', 'median_apartment_rent_per_week',
       'commute_time', 'num_of_rooms', 'inverse_cbd_distance'],
      dtype='object')


In [198]:
def classify_property_type_rf(train_df, test_df, features, target='type'):
    # Encode target labels
    le = LabelEncoder()
    train_df['type_encoded'] = le.fit_transform(train_df[target])
    test_df['type_encoded'] = le.transform(test_df[target])  

    X_train = train_df[features]
    y_train = train_df['type_encoded']
    X_test = test_df[features]
    y_test = test_df['type_encoded']

    # Train Random Forest
    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=15,
        random_state=42,
        class_weight='balanced'
    )
    model.fit(X_train, y_train)

    # Predict and evaluate
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    train_f1 = f1_score(y_train, train_preds, average='weighted',zero_division=1)
    test_f1 = f1_score(y_test, test_preds, average='weighted',zero_division=1)

    print(f"Train F1 Score: {train_f1:.3f}")
    print(f"Test F1 Score: {test_f1:.3f}")

    # Feature Importances
    importances = model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': features,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    print("\n📊 Feature Importances:")
    print(feature_importance_df)

    return train_f1, test_f1

In [199]:
classify_property_type_rf(train_df, test_df, features, target='type')

Train F1 Score: 0.990
Test F1 Score: 0.897

📊 Feature Importances:
                           Feature  Importance
4                    property_size    0.181403
11            inverse_cbd_distance    0.108862
10                    num_of_rooms    0.094361
9                     commute_time    0.081804
0                            price    0.080769
2                          num_bed    0.078829
8   median_apartment_rent_per_week    0.078578
5        suburb_median_house_price    0.072450
6       median_house_rent_per_week    0.069704
1                         num_bath    0.051766
3                      num_parking    0.051115
7    suburb_median_apartment_price    0.050360


(0.9900715110789223, 0.8973920998960474)